# LLM 向けの学習・評価用 Q&A データセット

質問と期待する回答の組を使うと、LLM に答え方を学ばせたり、学習後の能力を測ったりできます。読解では本文、論理推論では事実と規則、選択式では選択肢も入力に含まれます。以下の短い問題例は、各形式を説明するための例です。

## 学習用とベンチマーク用

- **学習用（train）**では、質問と模範回答を使ってモデルの重みを更新します。教師ありファインチューニング（SFT）では、模範回答のトークンを予測しやすくするように学習します。途中の解法も模範回答に含めれば、解き方も教師として与えられます。
- **検証用（validation / dev）**では、学習中のモデルや設定を比較します。
- **テスト用（test）**では、モデルの重みを固定し、問題を解かせて出力を正解と照合します。テスト問題とその正解は、学習や設定の選択に使わず、最終評価に取っておきます。

採点方法は問題形式に合わせます。数値問題なら最終的な数値、選択式なら選択肢、読解なら正解の文字列との完全一致や単語の重なりなどを使います。

## 代表的なデータセット

| データセット | 主な用途 | 入力と期待する回答 |
|---|---|---|
| **Stanford Alpaca** | 指示チューニング | 質問や依頼、必要に応じた追加情報 → 模範応答。Q&A のほか要約なども含みます。 [配布元](https://github.com/tatsu-lab/stanford_alpaca) |
| **SQuAD 2.0** | 読解の学習・評価 | Wikipedia の一節と質問 → 本文中の回答箇所、または本文からは回答できないという判定。 [配布元](https://rajpurkar.github.io/SQuAD-explorer/) |
| **GSM8K** | 算数推論の学習・評価 | 小学校レベルの文章題 → 途中の計算と最終的な数値。 [配布元](https://github.com/openai/grade-school-math) |
| **ProntoQA（PrOntoQA）** | 演繹推論の学習・評価 | 人工的に作った事実・規則と判定対象の命題 → 真偽と、その判定に至る推論。 [原論文](https://arxiv.org/abs/2210.01240) |
| **ProsQA** | 分岐のある論理推論の学習・評価 | 事実・規則と二つの候補 → 与えられた規則をたどって到達できる候補。 [Coconut 原論文 付録A](https://arxiv.org/html/2412.06769v3#A1) |
| **MMLU** | 多分野の知識・問題解決能力の評価 | 数学、歴史、法律など57分野の問題 → 正しい選択肢。 [原論文](https://arxiv.org/abs/2009.03300) |

## 算数と論理推論の問題例

### GSM8K の形式

**質問**　4か月に1回、100ドルで芝を刈ると、年間の費用はいくらですか？

**解法と答え**　$12/4=3$ 回なので、$3\times100=300$ ドルです。最終回答だけの学習なら「300ドル」を教師にし、解法も学ぶ場合は途中の計算を併せて与えます。

### ProntoQA の形式

**事実と規則**　Tom は terpus です。すべての terpus は brimpus です。すべての brimpus は lempus です。

**質問**　「Tom は lempus である」は真ですか？

**推論と答え**　Tom は terpus → brimpus → lempus と導けるため、真です。

### ProsQA の形式

**事実と規則**　Tom は terpus です。すべての terpus は brimpus であり、gerpus でもあります。すべての brimpus は lempus です。すべての gerpus は lorpus です。

**質問**　Tom は lempus ですか、それとも scrompus ですか？

**推論と答え**　terpus → brimpus → lempus の経路があるため、答えは lempus です。terpus → gerpus → lorpus も有効な推論ですが、この質問の二択にはつながりません。ProsQA では、このような分岐を含む有向非巡回グラフから問題を作ります。 [Coconut 原論文 付録A.2](https://arxiv.org/html/2412.06769v3#A1.SS2)

## Coconut 論文での使い方

算数実験では、合成した算数問題で学習し、GSM8K のテスト1,319問で評価しました。ProntoQA と ProsQA はそれぞれ学習用・検証用・テスト用に分けて使いました。論文では、通常の CoT と潜在表現を使う Coconut を、最終回答の正答率や生成トークン数で比較しています。 [Coconut 原論文 §5・付録A](https://arxiv.org/html/2412.06769v3)

Coconut の学習方法と実験結果は、[Coconut 補足ノート](8300.0170.Coconut.ipynb)で解説しています。

## Q&A を使った SFT 実習

[一次方程式の SFT 実習](8300.2001.SFT.linear_equations.ipynb)では、英語の Q&A でモデルを学習し、途中式を教師に含める場合と答えだけの場合を共通のテストで比較します。